# Proyecto de Análisis y Predicción de Siniestralidad Vial
**Equipo:** The Outliers

## 1. Objetivo del Análisis
El objetivo principal de este proyecto es desarrollar y comparar modelos de Machine Learning (Regresión Lineal Múltiple y Random Forest) capaces de predecir la probabilidad de que un accidente vial tenga víctimas fatales en base a factores ambientales y temporales.

Se busca identificar qué variables externas inciden con mayor peso en la gravedad y letalidad de los accidentes para generar un modelo predictivo robusto, cuyas métricas de rendimiento (RMSE, MAE, R²) serán registradas en una base de datos documental (MongoDB Atlas) para su auditoría y comparación.


## 2. Hipótesis Iniciales
Antes de la fase de modelado, establecemos las siguientes hipótesis rectoras que serán evaluadas empíricamente:

* **Hipótesis 1 (Factor Climático):** Las precipitaciones (lluvia) tienen una correlación positiva con el aumento en la probabilidad de que un siniestro sea fatal, debido a la reducción de visibilidad, la pérdida de adherencia en la calzada y un menor control del vehículo.
* **Hipótesis 2 (Efecto Calendario):** Existe una marcada estacionalidad en la gravedad de los siniestros. La probabilidad de que un accidente resulte en una víctima fatal durante los fines de semana y feriados es significativamente mayor que en los días laborables, probablemente debido a los cambios en la dinámica de movilidad urbana (ej. mayores velocidades en vías despejadas o conducción nocturna).
* **Hipótesis 3 (Rendimiento Algorítmico):** Dada la probable existencia de relaciones no lineales entre las variables (ej. el efecto combinado de lluvia intensa durante un fin de semana), se espera que el modelo de ensamble basado en árboles (Random Forest) supere al modelo paramétrico (Regresión Lineal) en las métricas de evaluación.

## 3. Importación de librerías

In [ ]:
%pip install pandas numpy scikit-learn pymongo plotly python-dotenv holidays matplotlib seaborn dash
import requests
import pandas as pd
import numpy as np
import holidays
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from datetime import datetime
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# MongoDB
import os
from dotenv import load_dotenv
from pymongo import MongoClient

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print('Librerías importadas correctamente.')

Note: you may need to restart the kernel to use updated packages.
Librerías importadas correctamente.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\pim\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [110]:
# Valores por defecto (Papermill los va a reemplazar desde afuera)
SEMILLA = 42
N_ESTIMADORES = 100
MAX_PROFUNDIDAD = 10

## 4. Carga y limpieza del dataset

In [111]:
# Guardas el enlace "Raw" en una variable (reemplaza esto con tu URL real)
url_github = 'https://raw.githubusercontent.com/Marian2057/The_Outliers/refs/heads/main/Proyecto/Data/siniestros_viales_victimas.csv'

# Pandas lee la URL directamente de la misma forma que un archivo local
try:
    tabla = pd.read_csv('siniestros_viales_victimas.csv', sep=';', low_memory=False)
    print(f'Cargado desde archivo local. Filas: {len(tabla):,}')
except FileNotFoundError:
    tabla = pd.read_csv(url_github, sep=';', low_memory=False)
    print(f'Cargado desde GitHub. Filas: {len(tabla):,}')

Cargado desde archivo local. Filas: 1,048,575


In [112]:
# Ver la cantidad de filas, columnas y los tipos de datos actuales
tabla.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1048575 entries, 0 to 1048574
Data columns (total 9 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   id_siniestro                 62076 non-null  object 
 1   fecha_siniestro              62076 non-null  object 
 2   anio_siniestro               62076 non-null  float64
 3   modo_desplazamiento_victima  62076 non-null  object 
 4   sexo_victima                 62076 non-null  object 
 5   edad_victima                 62076 non-null  object 
 6   GRAVEdad_victima             62076 non-null  object 
 7   rol_victima                  61865 non-null  object 
 8   fecha_fallecimiento_victima  610 non-null    object 
dtypes: float64(1), object(8)
memory usage: 72.0+ MB


In [113]:
# Ver un resumen estadístico de las columnas numéricas
tabla.describe(include='all')

,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
count,62076,62076,62076.000000,62076,62076,62076,62076,61865,610
unique,54064,2192,NaN,18,3,102,3,8,531
top,LC-2024-0243962,2024-12-17,NaN,SD,M,SD,LEVE,SD,2024-4-4
freq,26,86,NaN,21874,33666,16755,58995,49983,4
mean,NaN,NaN,2021.636784,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,1.779534,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,2019.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,2020.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,2022.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,2023.000000,NaN,NaN,NaN,NaN,NaN,NaN


In [114]:
# Contar cuántos valores nulos hay por columna
tabla.isnull().sum()

id_siniestro                    986499
fecha_siniestro                 986499
anio_siniestro                  986499
modo_desplazamiento_victima     986499
sexo_victima                    986499
edad_victima                    986499
GRAVEdad_victima                986499
rol_victima                     986710
fecha_fallecimiento_victima    1047965
dtype: int64

1. Eliminar las filas vacías
- No tiene sentido procesar un millón de filas vacías. Lo más seguro es eliminar cualquier fila que no tenga un identificador de siniestro.

In [115]:
# Eliminar filas donde 'id_siniestro' sea nulo
tabla.dropna(subset=['id_siniestro'], inplace=True)

# Verificar cuántas filas quedaron
print("Total de filas tras la limpieza:", len(tabla))

Total de filas tras la limpieza: 62076


2. Tratar los valores "SD" (Sin Datos)
- Como vimos antes, es probable que la edad o las fechas tengan el texto "SD". Vamos a estandarizarlo a un nulo real (NaN) que Pandas pueda entender.

In [116]:
# Reemplazar "SD" por NaN (nulo real de numpy/pandas) en todo el DataFrame
tabla.replace('SD', np.nan, inplace=True)

# Ahora sí, si vuelves a hacer tabla.isnull().sum(), verás los nulos reales

3. Corregir los Tipos de Datos (Dtypes)
- Darle a cada columna el formato correcto. Esto es crucial si luego vas a conectar estos datos a herramientas de visualización para armar tus tableros.

In [117]:
# 1. Convertir fechas a formato datetime
tabla['fecha_siniestro'] = pd.to_datetime(tabla['fecha_siniestro'], errors='coerce')
tabla['fecha_fallecimiento_victima'] = pd.to_datetime(tabla['fecha_fallecimiento_victima'], errors='coerce')

# 2. Convertir edad a número (estaba como texto 'object')
tabla['edad_victima'] = pd.to_numeric(tabla['edad_victima'], errors='coerce')
# Convertir la columna a formato entero que acepta nulos
tabla['edad_victima'] = tabla['edad_victima'].astype('Int64')

# 3. Convertir el año a número entero 
# (estaba como float64 porque Pandas usa decimales cuando hay filas nulas)
tabla['anio_siniestro'] = tabla['anio_siniestro'].astype(int)

# 4. Normalización de columnas de texto (Mayúsculas y sin espacios extra)
columnas_texto = [
    'modo_desplazamiento_victima', 
    'sexo_victima', 
    'GRAVEdad_victima', 
    'rol_victima'
]

for col in columnas_texto:
    # Convertimos a string, pasamos a mayúsculas y borramos espacios al principio/final
    tabla[col] = tabla[col].astype(str).str.strip().str.upper()

# Opcional: Reemplazar el texto "NAN" (que se genera al convertir nulos a string) por el nulo real de nuevo
tabla.replace('NAN', np.nan, inplace=True)

In [118]:
tabla.head(5)

,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
0,LC-2019-0000647,2019-01-01,2019,MOTO,M,54,GRAVE,NaN,NaT
1,LC-2019-0000600,2019-01-01,2019,NaN,F,1,LEVE,NaN,NaT
2,LC-2019-0000136,2019-01-01,2019,NaN,F,21,LEVE,NaN,NaT
3,LC-2019-0000082,2019-01-01,2019,NaN,F,32,LEVE,NaN,NaT
4,LC-2019-0000194,2019-01-01,2019,NaN,F,33,LEVE,NaN,NaT


## 5. Enriquecer el dataset
### 5.1 Descargar datos de Open-Meteo

Usamos la [API histórica de Open-Meteo](https://open-meteo.com/) — gratuita y sin API key — para obtener temperatura media y precipitaciones diarias de Buenos Aires (2019–2024).

In [119]:
# Coordenadas exactas de Buenos Aires
lat = -34.6037
lon = -58.3816

# Ajusta fecha_fin hasta el último día que tengas en tu dataset de accidentes
fecha_inicio = '2019-01-01'
fecha_fin = '2024-12-31' 

# Llamada a la API pidiendo temperatura media y suma de precipitaciones diarias
url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={fecha_inicio}&end_date={fecha_fin}&daily=temperature_2m_mean,precipitation_sum&timezone=America%2FSao_Paulo"

respuesta = requests.get(url).json()

# Convertir el resultado directamente a un DataFrame
df_clima = pd.DataFrame(respuesta['daily'])

# Acomodar los tipos de datos y nombres para que coincidan con nuestra tabla
df_clima['time'] = pd.to_datetime(df_clima['time'])
df_clima.rename(columns={
    'time': 'fecha', 
    'temperature_2m_mean': 'temp_media', 
    'precipitation_sum': 'precipitacion'
}, inplace=True)

print("Datos del clima descargados:")
print(df_clima.head())

Datos del clima descargados:
       fecha  temp_media  precipitacion
0 2019-01-01        25.9            5.3
1 2019-01-02        26.0           30.0
2 2019-01-03        20.3            0.1
3 2019-01-04        22.4            0.0
4 2019-01-05        24.5            0.0


### 5.2 Construir la variable objetivo y unir con el Clima

La **variable objetivo** es `probabilidad_fatalidad`: la proporción de víctimas fatales por día, calculada como el promedio de `es_fatal` (1 si hubo fecha de fallecimiento, 0 si no) para cada fecha. Esto produce un valor continuo entre 0.0 y 1.0, apto para modelos de regresión con métricas RMSE, MAE y R².

In [120]:
# 1. Crear una columna binaria que indique si el accidente tuvo una víctima fatal
# Asumimos fatalidad si hay una fecha de fallecimiento registrada
tabla['es_fatal'] = tabla['fecha_fallecimiento_victima'].notna().astype(int)

# 2. Agrupar por fecha calculando la probabilidad (promedio de es_fatal por día)
df_diario = tabla.groupby('fecha_siniestro').agg(
    probabilidad_fatalidad=('es_fatal', 'mean') # Variable continua entre 0.0 y 1.0
).reset_index()

# 3. Renombrar y asegurar formato datetime para el cruce con el clima
df_diario.rename(columns={'fecha_siniestro': 'fecha'}, inplace=True)
df_diario['fecha'] = pd.to_datetime(df_diario['fecha'])

# 4. Cruce de datos (Left Join) con df_clima que ya tienen procesado
df_final = pd.merge(df_diario, df_clima, on='fecha', how='left')

### 5.3 Extracción de Características Temporales
Es necesario descomponer la variable fecha en componentes cíclicos y binarios que aporten capacidad predictiva sobre la varianza en la frecuencia de los siniestros.

In [121]:
# Extracción de componentes base
df_final['mes'] = df_final['fecha'].dt.month
df_final['dia_semana'] = df_final['fecha'].dt.dayofweek  # 0=Lunes, 6=Domingo
df_final['es_fin_de_semana'] = df_final['dia_semana'].apply(lambda x: 1 if x >= 5 else 0)

# Integración de la variable de feriados
feriados_arg = holidays.AR(years=range(2019, 2025))
df_final['es_feriado'] = df_final['fecha'].apply(lambda x: 1 if x in feriados_arg else 0)

''' 
# Eliminar la variable datetime original, reteniendo la variable objetivo y las features
df_final = df_final.drop(columns=['fecha'])'''

# Verificar las variables predictoras resultantes
df_final.head()

,fecha,probabilidad_fatalidad,temp_media,precipitacion,mes,dia_semana,es_fin_de_semana,es_feriado
0,2019-01-01,0.0,25.9,5.3,1,1,0,1
1,2019-01-02,0.0,26.0,30.0,1,2,0,0
2,2019-01-03,0.0,20.3,0.1,1,3,0,0
3,2019-01-04,0.0,22.4,0.0,1,4,0,0
4,2019-01-05,0.0,24.5,0.0,1,5,1,0


## 6. Entrenar los modelos
### 6.1 Partición del Dataset (Train/Test Split)
Para evaluar la capacidad de generalización del modelo, dividimos la matriz de datos en dos conjuntos:
* **Entrenamiento (Train):** 80% de los datos para ajustar los parámetros del modelo.
* **Prueba (Test):** 20% de datos "invisibles" para la evaluación final.
Se utiliza una semilla (`random_state=42`) para garantizar la reproducibilidad del experimento en distintas ejecuciones.

In [122]:
# 1. Separación de Variables (Features vs Target)
columnas_modelo = ['temp_media', 'precipitacion', 'mes', 'dia_semana', 'es_fin_de_semana', 'es_feriado']
X = df_final[columnas_modelo]

# CORRECCIÓN CRÍTICA: Asignación de la nueva variable objetivo
y = df_final['probabilidad_fatalidad'] 

### 6.2 Preprocesamiento de Variables Independientes (Features)
Los algoritmos paramétricos como la Regresión Lineal son sensibles a las escalas y tipos de datos. Diseñamos un `ColumnTransformer` para aplicar el tratamiento estadístico adecuado a cada tipo de variable sin incurrir en fuga de datos (*data leakage*):
* **Variables Numéricas Continuas:** Se aplica `StandardScaler` para centrar la media en 0 y la varianza en 1, evitando que variables con magnitudes altas dominen el modelo.
* **Variables Categóricas (Cíclicas):** Se aplica `OneHotEncoder` para evitar que el algoritmo asuma un orden de magnitud matemático en los meses o días de la semana. Usamos `drop='first'` para evadir la colinealidad perfecta (la trampa de las variables ficticias).
* **Variables Binarias:** Ingresan directamente al modelo (`passthrough`).

In [123]:
# 2. Partición Train/Test (80% entrenamiento, 20% prueba)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEMILLA)

print(f"Dimensiones de X_train: {X_train.shape}")
print(f"Dimensiones de y_train: {y_train.shape}")


Dimensiones de X_train: (1753, 6)
Dimensiones de y_train: (1753,)


### 6.3 Construcción del Pipeline — Modelo 1: Regresión Lineal (Baseline)
Se ensambla una arquitectura de `Pipeline` de Scikit-Learn que integra secuencialmente el motor de preprocesamiento y el algoritmo estimador. 
Como modelo base de experimentación, implementamos una **Regresión Lineal Múltiple**. Esto nos proporcionará las métricas iniciales de rendimiento contra las cuales compararemos modelos de ensamble más complejos (ej. Random Forest).

In [124]:
# 3. Definición de los grupos de columnas
num_features = ['temp_media', 'precipitacion']
cat_features = ['mes', 'dia_semana']
bin_features = ['es_fin_de_semana', 'es_feriado']

In [125]:
# 4. Configuración del ColumnTransformer
preprocesador = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        # drop='first' evita la trampa de las variables ficticias (colinealidad perfecta)
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features),
        ('bin', 'passthrough', bin_features)
    ])

In [126]:
# 5. Ensamblado del Pipeline con el algoritmo de Regresión Lineal
pipeline_regresion = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', LinearRegression())
])

# 6. Entrenamiento del modelo
pipeline_regresion.fit(X_train, y_train)

print("Pipeline entrenado correctamente.")

Pipeline entrenado correctamente.


### 6.4 Construcción del Pipeline — Modelo 2: Random Forest

El **Random Forest** es un modelo de ensamble que construye múltiples árboles de decisión y promedia sus predicciones. Sus ventajas sobre la Regresión Lineal son:
* Captura relaciones **no lineales** entre variables.
* Es robusto ante outliers y no requiere supuestos de distribución.
* Ofrece una medida de **importancia de variables** que permite interpretar cuáles features son más predictivas.

Usamos el **mismo `preprocesador`** definido antes, garantizando que ambos modelos reciban exactamente el mismo input.

In [127]:
# Ensamblado del Pipeline con el algoritmo Random Forest
# Se reutiliza el 'preprocesador' definido en la celda anterior
pipeline_rf = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('modelo', RandomForestRegressor(
        n_estimators=N_ESTIMADORES,
        max_depth=MAX_PROFUNDIDAD, 
        min_samples_split=5,
        random_state=SEMILLA,
        n_jobs=-1
    ))
])

# Entrenamiento del modelo
pipeline_rf.fit(X_train, y_train)

print("Pipeline de Random Forest entrenado correctamente.")

Pipeline de Random Forest entrenado correctamente.


## 7. Evaluación de Modelos

Generamos las predicciones sobre el conjunto de test (`X_test`) y calculamos las tres métricas obligatorias:

| Métrica | Interpretación |
|---------|---------------|
| **RMSE** | Error cuadrático medio. Penaliza más los errores grandes. |
| **MAE**  | Error absoluto promedio. Más interpretable. |
| **R²**   | Proporción de varianza explicada por el modelo (0–1). |

In [128]:
# 1. Generar predicciones con ambos modelos
y_pred_lr = pipeline_regresion.predict(X_test)
y_pred_rf = pipeline_rf.predict(X_test)

# 2. Función para calcular y mostrar métricas
def evaluar_modelo(y_true, y_pred, nombre_modelo):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    print(f"--- Métricas {nombre_modelo} ---")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"R²: {r2:.4f}\n")
    
    return {"Modelo": nombre_modelo, "RMSE": rmse, "MAE": mae, "R2": r2,
            "timestamp": datetime.now().isoformat()}

# Calculamos las métricas
metricas_lr = evaluar_modelo(y_test, y_pred_lr, "Regresión Lineal")
metricas_rf = evaluar_modelo(y_test, y_pred_rf, "Random Forest")

# 3. Tabla comparativa
df_metricas = pd.DataFrame([metricas_lr, metricas_rf])[['Modelo','RMSE','MAE','R2']]
print('=' * 50)
print('       COMPARACIÓN DE MODELOS')
print('=' * 50)
print(df_metricas.to_string(index=False))



--- Métricas Regresión Lineal ---
RMSE: 0.0253
MAE: 0.0171
R²: 0.0158

--- Métricas Random Forest ---
RMSE: 0.0270
MAE: 0.0178
R²: -0.1227

       COMPARACIÓN DE MODELOS
          Modelo     RMSE      MAE        R2
Regresión Lineal 0.025264 0.017098  0.015833
   Random Forest 0.026983 0.017782 -0.122667


## 8. Almacenamiento en MongoDB (NoSQL)

Guardamos tres colecciones tal como requiere la consigna:
- **`datos_entrada`**: dataset limpio y enriquecido.
- **`resultados_modelo`**: predicciones y métricas de cada modelo.
- **`configuracion_modelo`**: hiperparámetros y estructura del pipeline.

> **Configuración:** Reemplazá `MONGO_URI` con tu cadena de conexión de MongoDB Atlas.  
> La encontrás en Atlas → Connect → Drivers → Python.  
> Forma: `mongodb+srv://<usuario>:<password>@cluster0.xxxxx.mongodb.net/`

In [129]:
# ─── Configuración de conexión ────────────────────────────────────────────────
# Esto lee el archivo .env que creaste recién
load_dotenv()

# Acá Python trae la contraseña de forma segura sin mostrarla
MONGO_URI = os.getenv('MONGO_URI')
DB_NAME   = 'siniestros_viales'

try:
    cliente = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
    cliente.server_info()  
    db = cliente[DB_NAME]
    print(f'¡Conectado a MongoDB Atlas con éxito! — base de datos: "{DB_NAME}"')
    MONGO_OK = True
except Exception as e:
    print(f'MongoDB no disponible: {e}')
    MONGO_OK = False

MongoDB no disponible: module 'lib' has no attribute 'X509_get_default_cert_dir_env'


In [130]:
if MONGO_OK:
    # ── Colección 1: datos_entrada ────────────────────────────────────────────
    col_entrada = db['datos_entrada']
    col_entrada.drop()  # Evita duplicados en re-ejecuciones

    registros_entrada = df_final.copy()
    registros_entrada['fecha'] = registros_entrada['fecha'].dt.strftime('%Y-%m-%d')
    col_entrada.insert_many(registros_entrada.to_dict(orient='records'))
    print(f'datos_entrada:      {col_entrada.count_documents({}):,} documentos guardados.')

    # ── Colección 2: resultados_modelo ────────────────────────────────────────
    col_resultados = db['resultados_modelo']
    col_resultados.drop()

    predicciones_doc = pd.DataFrame({
        'y_real':   y_test.values.tolist(),
        'pred_lr':  y_pred_lr.round(4).tolist(),
        'pred_rf':  y_pred_rf.round(4).tolist(),
        'error_lr': (y_test.values - y_pred_lr).round(4).tolist(),
        'error_rf': (y_test.values - y_pred_rf).round(4).tolist(),
    })
    col_resultados.insert_many(predicciones_doc.to_dict(orient='records'))

    # Guardar también el resumen de métricas
    db['metricas_modelo'].drop()
    db['metricas_modelo'].insert_many([metricas_lr, metricas_rf])
    print(f'resultados_modelo:  {col_resultados.count_documents({})} predicciones guardadas.')

    # ── Colección 3: configuracion_modelo ─────────────────────────────────────
    col_config = db['configuracion_modelo']
    col_config.drop()

    configs = [
        {
            'modelo':       'Regresión Lineal',
            'algoritmo':    'LinearRegression',
            'params':       pipeline_regresion.named_steps['modelo'].get_params(),
            'preproceso':   'ColumnTransformer (StandardScaler + OneHotEncoder + passthrough)',
            'features':     columnas_modelo,
            'target':       'probabilidad_fatalidad',
            'train_size':   0.8,
            'test_size':    0.2,
            'random_state': 42,
            'timestamp':    datetime.now().isoformat()
        },
        {
            'modelo':       'Random Forest',
            'algoritmo':    'RandomForestRegressor',
            'params':       {k: v for k, v in pipeline_rf.named_steps['modelo'].get_params().items()
                             if not callable(v)},
            'preproceso':   'ColumnTransformer (StandardScaler + OneHotEncoder + passthrough)',
            'features':     columnas_modelo,
            'target':       'probabilidad_fatalidad',
            'train_size':   0.8,
            'test_size':    0.2,
            'random_state': 42,
            'timestamp':    datetime.now().isoformat()
        }
    ]
    col_config.insert_many(configs)
    print(f'configuracion_modelo: {col_config.count_documents({})} documentos guardados.')

## 9. Visualización de Resultados

In [131]:
# ── Gráfico 1: Comparación de Error (RMSE y MAE) entre Modelos ────────────────

# Prepara los datos para este gráfico
df_metricas_plot = pd.DataFrame([metricas_lr, metricas_rf])

# 1. Creamos el gráfico con Plotly Express
fig_errores = px.bar(
    df_metricas_plot,
    x='Modelo',
    y=['RMSE', 'MAE'],
    barmode='group',
    title='Comparación de Error (RMSE y MAE) entre Modelos',
    labels={'value': 'Valor del Error', 'variable': 'Métrica', 'Modelo': ''},
    color_discrete_sequence=['#4A90D9', '#94A3B8'], # Colores del dashboard (Azul y Gris)
    text_auto='.5f' 
)

# 2. DISEÑO UNIFICADO: Aplicamos el fondo oscuro y letras claras del Dashboard
fig_errores.update_layout(
    template=None,
    paper_bgcolor='#1A1D27', # Fondo de tarjeta oscuro
    plot_bgcolor='#1A1D27',  # Fondo de gráfico oscuro
    
    title_font_size=20,
    title_font_family='Arial',
    font_color='#F1F5F9',    # Letras principales en blanco/gris claro
    
    legend_title_text='Métrica',
    legend_yanchor='top',
    legend_y=0.98,
    legend_xanchor='right',
    legend_x=1.06,
    hovermode='x unified' 
)

# 3. Forzamos los ejes y las líneas de guía para que sean tenues y claros
fig_errores.update_xaxes(
    tickfont=dict(size=13, color='#94A3B8'), # Números del eje X claros
    title_font=dict(color='#94A3B8')
)
fig_errores.update_yaxes(
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)',      # Grilla interna hiper tenue para fondo oscuro
    tickfont=dict(color='#94A3B8'),          # Números del eje Y claros
    title_text='Valor del Error (Menor es mejor)',
    title_font=dict(color='#94A3B8')
)

fig_errores.update_traces(textposition='outside', textfont_size=11)

# Mostramos el gráfico interactivo ya con el look oscuro
fig_errores.show()

El gráfico presenta una comparación directa de los errores de predicción (RMSE y MAE) entre el modelo de Regresión Lineal y el algoritmo de Random Forest, evaluados sobre el conjunto de test. Dado que ambas métricas miden el desvío de las predicciones respecto a los valores reales, un valor menor indica un mejor rendimiento.

1. Interpretación de las Métricas
MAE (Error Absoluto Medio): La Regresión Lineal presenta un MAE de 0.01710 frente al 0.01778 de Random Forest. Esto significa que, en promedio, las predicciones de la Regresión Lineal se equivocan por un margen ligeramente menor.

RMSE (Error Cuadrático Medio): En ambas alternativas el RMSE es mayor que el MAE (LR: 0.02526 vs. RF: 0.02698). Esto es un comportamiento normal en ciencia de datos, ya que el RMSE penaliza con mayor severidad los errores grandes o valores atípicos (outliers).

2. Conclusión Principal
Contrario a la intuición inicial de que los modelos basados en árboles siempre superan a los lineales, para este conjunto de datos específico de siniestralidad vial, la Regresión Lineal demostró una capacidad de generalización ligeramente superior, manteniendo tanto el error promedio como la penalización por grandes desvíos por debajo de los niveles de Random Forest.

In [ ]:
# ── Gráfico 2: Real vs Predicho — ambos modelos ───────────────────────────────


# 1. Calculamos los límites globales para que la línea roja cruce perfecto
min_val = min(y_test.min(), y_pred_lr.min(), y_pred_rf.min())
max_val = max(y_test.max(), y_pred_lr.max(), y_pred_rf.max())
lims = [min_val, max_val]

# 2. Creamos la estructura de 1 fila y 2 columnas (Subplots)
fig_real_predicho = make_subplots(
    rows=1, cols=2, 
    subplot_titles=(
        f'Regresión Lineal (R²={metricas_lr["R2"]:.4f})', 
        f'Random Forest (R²={metricas_rf["R2"]:.4f})'
    )
)

# --- SUBPLOT 1: Regresión Lineal (Izquierda) ---
fig_real_predicho.add_trace(
    go.Scatter(
        x=y_test, y=y_pred_lr, mode='markers',
        marker=dict(color='#4A90D9', opacity=0.5, size=6), # Azul claro
        name='Predicciones LR'
    ),
    row=1, col=1
)
fig_real_predicho.add_trace(
    go.Scatter(
        x=lims, y=lims, mode='lines',
        line=dict(color='#F87171', dash='dash', width=2), # Coral para resaltar el error
        name='Predicción perfecta'
    ),
    row=1, col=1
)

# --- SUBPLOT 2: Random Forest (Derecha) ---
fig_real_predicho.add_trace(
    go.Scatter(
        x=y_test, y=y_pred_rf, mode='markers',
        marker=dict(color='#2DD4BF', opacity=0.5, size=6), # Verde Teal
        name='Predicciones RF'
    ),
    row=1, col=2
)
fig_real_predicho.add_trace(
    go.Scatter(
        x=lims, y=lims, mode='lines',
        line=dict(color='#F87171', dash='dash', width=2),
        showlegend=False
    ),
    row=1, col=2
)

# 3. DISEÑO UNIFICADO OSCURO
fig_real_predicho.update_layout(
    title_text='Real vs. Predicho — Conjunto de Test',
    title_font_size=20,
    title_font_family='Arial',
    
    font_color='#F1F5F9',     # Letras principales claras
    paper_bgcolor='#1A1D27',  # Fondo tarjeta oscuro
    plot_bgcolor='#1A1D27',   # Fondo gráfico oscuro
    
    height=500,
    width=1100,
    showlegend=True,
    legend=dict(font=dict(color='#94A3B8')),
    hovermode='closest'
)

# 4. Ajustes de legibilidad para ejes y grillas tenues
fig_real_predicho.update_xaxes(
    title_text='Probabilidad real', 
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)
fig_real_predicho.update_yaxes(
    title_text='Probabilidad predicha', 
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)

# Truco para forzar el color claro en los subtítulos de los subplots
for annotation in fig_real_predicho['layout']['annotations']:
    annotation['font'] = dict(color='#F1F5F9')

# Mostramos el gráfico interactivo
fig_real_predicho.show()

Análisis de la Relación Real vs. Predicho
El gráfico de dispersión (Scatter Plot) entre los valores reales y predichos expone la dificultad de ambos algoritmos para capturar la variabilidad de la siniestralidad vial. La línea de puntos roja representa la "predicción perfecta" (donde el valor real y el predicho coinciden).
Visualmente, se observa que ambos modelos sufren de un sesgo de subestimación: las predicciones se agrupan de forma casi horizontal en el rango inferior (entre 0.00 y 0.03), siendo incapaces de proyectar los casos donde la probabilidad real escala hasta 0.20.

Diagnóstico del Coeficiente de Determinación (R²)

•	Regresión Lineal (R² = 0.0158): Explica apenas el 1.58% de la varianza del fenómeno. El modelo lineal no encuentra una estructura clara, distribuyendo sus predicciones en una nube difusa en la base del gráfico.

•	Random Forest (R² = -0.1227): Al arrojar un valor negativo sobre el conjunto de test, se evidencia un problema crítico de sobreajuste (overfitting) en el entrenamiento o una severa penalización por el desbalance de los datos. El modelo comete errores sistemáticos más grandes que la varianza natural de la variable objetivo.


In [133]:
# ── Gráfico 3: Importancia de variables — Random Forest ───────────────────────

# 1. Recuperar los nombres de features tras el OneHotEncoder
ohe_feature_names = (
    pipeline_rf
    .named_steps['preprocesamiento']
    .named_transformers_['cat']
    .get_feature_names_out(cat_features)
    .tolist()
)
all_feature_names = num_features + ohe_feature_names + bin_features

# 2. Calcular las importancias y ordenar de menor a mayor
importancias = pd.Series(
    pipeline_rf.named_steps['modelo'].feature_importances_,
    index=all_feature_names
).sort_values(ascending=True)

# Tomamos las 10 más importantes
df_importancias = importancias.tail(10).reset_index()
df_importancias.columns = ['Variable', 'Importancia']

# 3. Creamos el gráfico de barras horizontales interactivo
fig_importancias = px.bar(
    df_importancias,
    x='Importancia',
    y='Variable',
    orientation='h',
    title='Top 10 Variables más importantes — Random Forest',
    labels={'Importancia': 'Importancia Relativa', 'Variable': ''},
    color_discrete_sequence=['#4A90D9'], # Azul vibrante para que destaque en fondo oscuro
    text_auto='.3f' 
)

# 4. DISEÑO UNIFICADO OSCURO
fig_importancias.update_layout(
    template=None,
    paper_bgcolor='#1A1D27', # Fondo tarjeta oscuro
    plot_bgcolor='#1A1D27',  # Fondo gráfico oscuro
    
    title_font_size=20,
    title_font_family='Arial',
    font_color='#F1F5F9',    # Letras claras
    
    height=500,
    width=900,
    showlegend=False, 
    margin=dict(l=150, r=50) # Mantenemos el margen vital para los nombres de las variables
)

# 5. Ajustes de legibilidad para ejes y grillas tenues
fig_importancias.update_xaxes(
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)
fig_importancias.update_yaxes(
    showgrid=False, # Sin grilla horizontal para un diseño más limpio
    color='#94A3B8',
    tickfont=dict(size=12, color='#94A3B8')
)

# Posicionamos los números por fuera de la barra y forzamos su color claro
fig_importancias.update_traces(
    textposition='outside', 
    textfont=dict(size=11, color='#F1F5F9')
)

# Mostramos la visualización interactiva
fig_importancias.show()

Este gráfico permite abrir la "caja negra" del algoritmo Random Forest para entender qué variables aportaron mayor ganancia de información al modelo al momento de predecir la siniestralidad vial.

La suma de la importancia de todas las variables del proyecto equivale a 1.00 (100%), y el gráfico muestra el top 10 de las características con mayor peso:

Dominio Climático: Los factores meteorológicos lideran de forma contundente la capacidad predictiva del modelo. La variable temp_media concentra el 41.6% de la importancia, seguida por precipitacion con el 21.1%. En conjunto, el clima explica más del 62% de las decisiones del algoritmo.

Factores Temporales y Calendario: Las variables asociadas al tiempo (como días específicos de la semana o meses del año) muestran un impacto secundario. Por ejemplo, dia_semana_6 (Sábado) tiene un peso del 4.4%, mientras que variables como es_fin_de_semana o es_feriado rondan apenas el 3.7% - 4.0%.

Conclusión Analítica
El análisis revela que para este modelo el contexto ambiental (si hace frío/calor o si hay lluvias) es un predictor muchísimo más potente de incidentes viales que el factor puramente cronológico (el día de la semana o el mes del año). Las variables climáticas seleccionadas por el equipo aportan la mayor cantidad de "señal" al algoritmo.

In [134]:
# ── Gráfico 4: Evolución de la probabilidad de fatalidad mensual ──────────────

# 1. Agrupamos los datos por mes de forma idéntica a tu lógica original
df_mensual = (
    df_final
    .set_index('fecha')['probabilidad_fatalidad']
    .resample('ME')
    .mean()
    .reset_index()
)

# 2. Creamos el gráfico de línea interactivo con Plotly Express
fig_linea = px.line(
    df_mensual,
    x='fecha',
    y='probabilidad_fatalidad',
    title='Probabilidad de fatalidad promedio mensual (2019–2024)',
    labels={'fecha': 'Fecha', 'probabilidad_fatalidad': 'Probabilidad promedio'},
    markers=True 
)

# 3. Forzamos el color azul vibrante para que resalte en el fondo oscuro
fig_linea.update_traces(
    line_color='#4A90D9', 
    marker=dict(size=6, color='#4A90D9')
)

# 4. DISEÑO UNIFICADO OSCURO
fig_linea.update_layout(
    template=None,
    paper_bgcolor='#1A1D27', 
    plot_bgcolor='#1A1D27',
    
    title_font_size=20,
    title_font_family='Arial',
    font_color='#F1F5F9',
    
    height=450,
    width=1100
)

# 5. Ajustes de legibilidad para los ejes (rotación de fechas y grillas tenues)
fig_linea.update_xaxes(
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8'),
    tickangle=30 
)
fig_linea.update_yaxes(
    title_text='Probabilidad de víctima fatal',
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)

# Mostramos la línea de tiempo interactiva
fig_linea.show()

Análisis de Evolución Temporal (Tendencia Mensual 2019–2024)
Este gráfico de serie temporal permite evaluar cómo varió la probabilidad promedio mensual de fatalidad estimada por el modelo a lo largo del tiempo. Al analizar la curva, se identifican tres comportamientos analíticos clave:

Pico Máximo Histórico (Principio de 2021): El gráfico registra su valor más alto de todo el período a comienzos de 2021, donde la probabilidad promedio mensual escala drásticamente superando el 0.041. Este comportamiento representa una anomalía severa en los datos que duplica el comportamiento habitual de la serie.

Picos Secundarios (Estacionalidad): Se observan comportamientos cíclicos o de corto plazo repetitivos. Hay un repunte notable a mediados de 2020 (alcanzando casi 0.027) y otro a principios de 2022 (rozando el 0.020). Estos picos recurrentes sugieren que existen meses específicos donde las condiciones del entorno elevan sistemáticamente el riesgo de fatalidad.

Tendencia a la Estabilización (2023–2024): A partir de 2023, la serie rompe la alta volatilidad de los años anteriores. Las fluctuaciones se vuelven mucho más atenuadas y predecibles, estabilizándose en una banda estrecha que oscila entre 0.005 y 0.017. La variabilidad disminuye notablemente hacia el final del período analizado.

Conclusión Teórica para el Informe
La fuerte oscilación entre 2020 y 2021 seguida por la estabilización en los años recientes (2023-2024) demuestra que el modelo es sensible a los cambios del entorno a nivel macroscópico. Para la gestión pública, este tablero interactivo permite identificar que, si bien el riesgo histórico tuvo picos críticos aislados, la tendencia actual del fenómeno se encuentra controlada y bajo una dinámica mucho más estable.

In [135]:
# ── Gráfico 5: Probabilidad de fatalidad por día de la semana ─────────────────

# 1. Agrupamos y preparamos los datos de forma idéntica a tu lógica
df_dias = df_final.groupby('dia_semana')['probabilidad_fatalidad'].mean().reset_index()
df_dias['Día'] = ['Lun', 'Mar', 'Mié', 'Jue', 'Vie', 'Sáb', 'Dom']
df_dias['Tipo de Día'] = ['Día Laboral'] * 5 + ['Fin de Semana'] * 2

# 2. Creamos el gráfico de barras con Plotly Express cambiando colores
fig_dias = px.bar(
    df_dias,
    x='Día',
    y='probabilidad_fatalidad',
    color='Tipo de Día', 
    title='Probabilidad de fatalidad promedio por día de la semana',
    labels={'probabilidad_fatalidad': 'Probabilidad media', 'Día': ''},
    # Nueva paleta: Gris claro para días laborales y Azul vibrante para fin de semana
    color_discrete_map={'Día Laboral': '#94A3B8', 'Fin de Semana': '#4A90D9'},
    text_auto='.3f' 
)

# 3. DISEÑO UNIFICADO OSCURO
fig_dias.update_layout(
    template=None,
    paper_bgcolor='#1A1D27',
    plot_bgcolor='#1A1D27',
    
    title_font_size=20,
    title_font_family='Arial',
    font_color='#F1F5F9',
    
    height=450,
    width=1150,
    
    # Configuración de la leyenda con color claro
    legend=dict(
        title_text='',
        font=dict(color='#94A3B8'),
        yanchor='top',
        y=1,
        xanchor='right',
        x=1.18 
    ),
    margin=dict(t=60, b= 40, l=150, r=150) 
)

# 4. Ajustes de legibilidad para ejes y grillas tenues
fig_dias.update_xaxes(
    color='#94A3B8',
    tickfont=dict(size=12, color='#94A3B8')
)
fig_dias.update_yaxes(
    title_text='Probabilidad media de víctima fatal',
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)

# Posicionamos los números por fuera de las barras y en color claro
fig_dias.update_traces(
    textposition='outside', 
    textfont=dict(size=11, color='#F1F5F9')
)

# Mostramos el gráfico interactivo
fig_dias.show()

**Análisis de Fatalidad por Día de la Semana**

Este gráfico analiza la probabilidad media de fatalidad estimada por el modelo, segmentando el comportamiento entre Días Laborales (Lunes a Viernes, en gris) y Fin de Semana (Sábado y Domingo, en azul).

**Comportamiento en Días Laborales (Lun a Vie):** El riesgo se mantiene en un piso relativamente estable y bajo, oscilando en una banda estrecha entre 0.008 (el mínimo, registrado los miércoles) y 0.011 (el pico de la semana laboral, registrado los jueves).

**Incremento el Fin de Semana (Sáb y Dom):** Se observa un salto marcado en la probabilidad a partir del sábado (0.014), alcanzando el máximo histórico de la serie los domingos con 0.015. Esto representa casi el doble del riesgo estimado para un día miércoles.

**Conclusión de los Datos**

Los datos confirman que el factor calendario introduce una variación sistemática en el riesgo. El marcado aumento concentrado en los días sábado y domingo valida estadísticamente por qué las variables de "fin de semana" ganaron peso dentro de las decisiones del Random Forest (como vimos en el gráfico de Importancia de Variables). Este patrón es clave para entender que el comportamiento del tránsito cambia de forma crítica durante los días de descanso.

In [136]:
# ── Gráfico 5B: Comparación Macrocategorías (Feriados Incluidos) ──────────────

# 1. Creamos una clasificación real sin pisarse
df_final['Categoría_Día'] = 'Día Laboral'
df_final.loc[df_final['es_fin_de_semana'] == 1, 'Categoría_Día'] = 'Fin de Semana'
df_final.loc[df_final['es_feriado'] == 1, 'Categoría_Día'] = 'Feriado' # El feriado manda si cae finde

# 2. Agrupamos por la nueva clasificación macro
df_macro = df_final.groupby('Categoría_Día')['probabilidad_fatalidad'].mean().reset_index()

# Ordenamos las barras de forma lógica para la vista
orden_macro = {'Día Laboral': 0, 'Fin de Semana': 1, 'Feriado': 2}
df_macro['orden'] = df_macro['Categoría_Día'].map(orden_macro)
df_macro = df_macro.sort_values('orden')

# 3. Creamos el gráfico con Plotly Express
fig_macro = px.bar(
    df_macro,
    x='Categoría_Día',
    y='probabilidad_fatalidad',
    color='Categoría_Día',
    title='Probabilidad de fatalidad promedio: Laborales vs. Fines de Semana vs. Feriados',
    labels={'probabilidad_fatalidad': 'Probabilidad media', 'Categoría_Día': ''},
    # Nueva paleta oscura: Gris para laboral, Azul vibrante para finde, Ámbar para feriado
    color_discrete_map={'Día Laboral': '#94A3B8', 'Fin de Semana': '#4A90D9', 'Feriado': '#F59E0B'},
    text_auto='.3f'
)

# 4. DISEÑO UNIFICADO OSCURO
fig_macro.update_layout(
    template=None,
    paper_bgcolor='#1A1D27',
    plot_bgcolor='#1A1D27',
    title_font_size=20,
    title_font_family='Arial',
    font_color='#F1F5F9',
    height=470,
    width=900,
    showlegend=False # No hace falta leyenda porque el nombre está abajo de cada barra
)

# 5. Ajustes de legibilidad para ejes y grillas tenues
fig_macro.update_xaxes(
    color='#94A3B8',
    tickfont=dict(size=13, color='#94A3B8')
)
fig_macro.update_yaxes(
    title_text='Probabilidad media de víctima fatal',
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)

# Forzamos que los números de arriba de las barras sean claros
fig_macro.update_traces(
    textposition='outside', 
    textfont_size=12, 
    textfont=dict(color='#F1F5F9')
)

fig_macro.show()

### Insights Adicionales

Más allá de las métricas de evaluación tradicionales (RMSE, MAE, $R^2$), un análisis de datos integral exige entender **por qué** los algoritmos encuentran dificultades para generalizar un patrón. 

El bajo rendimiento de los modelos predictivos en el conjunto de prueba no es necesariamente un defecto del pipeline, sino un síntoma de la **alta complejidad y no linealidad** de la siniestralidad vial. Las variables ambientales y de calendario interactúan de formas impredecibles que escapan a las regresiones simples.

Para entender este fenómeno a fondo, el equipo decidió cruzar las variables que el Random Forest identificó como más importantes. A continuación, se expone un hallazgo clave que demuestra cómo el riesgo de un día no laborable no es estático, sino que sufre un "efecto multiplicador" severo dependiendo de la época del año.

In [137]:
# 1. Instanciamos los feriados argentinos (tal como tenés en tu notebook)
feriados_arg = holidays.AR(years=range(2019, 2025))

# 2. En la tabla ORIGINAL (tabla), extraemos el nombre específico del feriado
# Usamos .date para que matchee perfecto con la librería holidays
tabla['nombre_feriado'] = tabla['fecha_siniestro'].apply(lambda x: feriados_arg.get(x.date()) if pd.notna(x) and x in feriados_arg else None)

# 3. Filtramos para quedarnos SOLO con los registros que ocurrieron en un feriado
df_solo_feriados = tabla[tabla['nombre_feriado'].notna()]

# 4. Agrupamos por el nombre del feriado y contamos cuántos accidentes ocurrieron en total
accidentes_por_feriado = df_solo_feriados.groupby('nombre_feriado').size().reset_index(name='total_accidentes')

# Ordenamos de mayor a menor y nos quedamos con el Top 10 para que no quede gigante
accidentes_por_feriado = accidentes_por_feriado.sort_values(by='total_accidentes', ascending=True).tail(10)

# 5. Armamos el gráfico de barras horizontales interactivo
fig_feriados = px.bar(
    accidentes_por_feriado,
    x='total_accidentes',
    y='nombre_feriado',
    orientation='h',
    title='Top 10 Feriados con Mayor Cantidad de Siniestros Acumulados (2019–2024)',
    labels={'total_accidentes': 'Cantidad Total de Siniestros', 'nombre_feriado': ''},
    color_discrete_sequence=['#F59E0B'], # Ámbar brillante unificado para feriados en fondo oscuro
    text_auto=True # Muestra el número exacto de accidentes al final de cada barra
)

# 6. DISEÑO UNIFICADO OSCURO
fig_feriados.update_layout(
    template=None,
    paper_bgcolor='#1A1D27', # Fondo tarjeta oscuro
    plot_bgcolor='#1A1D27',  # Fondo gráfico oscuro
    title_font_size=20,
    title_font_family='Arial',
    font_color='#F1F5F9',    # Letras principales claras
    height=500,
    width=1190,
    margin=dict(l=450, r=20, t=60, b=40) # Mantenemos el margen izquierdo generoso para que no se corten los nombres
)

# 7. Ajustes de legibilidad para ejes y grillas tenues
fig_feriados.update_xaxes(
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)
fig_feriados.update_yaxes(
    showgrid=False, # Al ser barras horizontales, se lee más limpio sin grillas verticales cruzadas
    color='#94A3B8',
    tickfont=dict(size=12, color='#94A3B8')
)

# Forzamos que los números totales al final de las barras se vean en color claro
fig_feriados.update_traces(
    textposition='outside', 
    textfont=dict(size=11, color='#F1F5F9')
)

fig_feriados.show()

In [138]:
# 1. Creamos una macro-categoría limpia para separar Días Comunes de Días Libres
df_final['tipo_dia_macro'] = df_final.apply(
    lambda r: 'Fin de Semana / Feriado' if (r['es_fin_de_semana'] == 1 or r['es_feriado'] == 1) else 'Día Laboral Común', 
    axis=1
)

# 2. Agrupamos por mes y por tipo de día para calcular la probabilidad promedio REAL
df_comportamiento = df_final.groupby(['mes', 'tipo_dia_macro'])['probabilidad_fatalidad'].mean().reset_index()

# Mapeamos los números a nombres de meses para que el eje X sea súper legible
meses_espanol = {1: 'Ene', 2: 'Feb', 3: 'Mar', 4: 'Abr', 5: 'May', 6: 'Jun', 
                 7: 'Jul', 8: 'Ago', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dic'}
df_comportamiento['Nombre_Mes'] = df_comportamiento['mes'].map(meses_espanol)

# 3. Creamos el gráfico de barras agrupadas por mes
fig_evidencia = px.bar(
    df_comportamiento,
    x='Nombre_Mes',
    y='probabilidad_fatalidad',
    color='tipo_dia_macro',
    barmode='group', # Pone las barras una al lado de la otra para comparar directo
    title='Evidencia Empírica: Probabilidad de Fatalidad por Mes y Tipo de Día',
    labels={'probabilidad_fatalidad': 'Probabilidad Media', 'Nombre_Mes': ''},
    # Nueva paleta oscura: Gris claro para días comunes y Azul vibrante para franco/fiestas
    color_discrete_map={'Día Laboral Común': '#94A3B8', 'Fin de Semana / Feriado': '#4A90D9'},
    text_auto='.3f'
)

# 4. DISEÑO UNIFICADO OSCURO
fig_evidencia.update_layout(
    template=None,
    paper_bgcolor='#1A1D27',
    plot_bgcolor='#1A1D27',
    title_font_size=20,
    title_font_family='Arial',
    font_color='#F1F5F9',
    height=500,
    width=1150,
    # Leyenda interna adaptada con tipografía clara
    legend=dict(
        title_text='', 
        font=dict(color='#94A3B8'),
        yanchor='top', y=1.10, xanchor='right', x=0.98
    )
)

# 5. Ajustes de legibilidad para ejes y grillas tenues
fig_evidencia.update_xaxes(
    color='#94A3B8',
    tickfont=dict(size=12, color='#94A3B8')
)
fig_evidencia.update_yaxes(
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)

# Forzamos los números de arriba de las barras a color claro
fig_evidencia.update_traces(
    textposition='outside', 
    textfont=dict(size=11, color='#F1F5F9')
)

fig_evidencia.show()

In [139]:
# 1. Usamos la 'tabla' original (que tiene los datos individuales de cada persona)
# Agrupamos por vehículo y calculamos la media de fatalidad y la cantidad total de casos
df_vehiculos = tabla.groupby('modo_desplazamiento_victima')['es_fatal'].agg(['mean', 'count']).reset_index()

# Renombramos las columnas para mayor claridad
df_vehiculos.rename(columns={'mean': 'probabilidad_fatalidad', 'count': 'total_involucrados'}, inplace=True)

# 2. Filtramos: nos quedamos solo con los medios de transporte que tengan más de 100 registros
# Esto evita que un vehículo raro con 1 solo accidente (que justo fue fatal) aparezca con 100% de letalidad
df_vehiculos = df_vehiculos[df_vehiculos['total_involucrados'] > 100]

# 3. Ordenamos de mayor a menor riesgo para que el gráfico cuente una historia clara
df_vehiculos = df_vehiculos.sort_values(by='probabilidad_fatalidad', ascending=False)

# 4. Armamos el gráfico de barras con Plotly Express
fig_vehiculos = px.bar(
    df_vehiculos,
    x='modo_desplazamiento_victima',
    y='probabilidad_fatalidad',
    title='Vulnerabilidad: Tasa de Letalidad según el Modo de Desplazamiento',
    labels={'probabilidad_fatalidad': 'Probabilidad media de fatalidad', 'modo_desplazamiento_victima': ''},
    color_discrete_sequence=['#4A90D9'], # Azul vibrante para modo oscuro
    text_auto='.3f'
)

# 5. DISEÑO UNIFICADO OSCURO
fig_vehiculos.update_layout(
    template=None,
    paper_bgcolor='#1A1D27',
    plot_bgcolor='#1A1D27',
    title_font_size=20,
    title_font_family='Arial',
    font_color='#F1F5F9',
    height=550,
    width=1050,
    margin=dict(t=60, b=70, l=80, r=40) # Mantenemos tus márgenes originales
)

# 6. Ajustes de legibilidad para ejes y grillas
fig_vehiculos.update_xaxes(
    color='#94A3B8',
    tickfont=dict(size=11, color='#94A3B8'), 
    tickangle=25 # Mantenemos la inclinación original para que se lean los vehículos
)
fig_vehiculos.update_yaxes(
    showgrid=True, 
    gridcolor='rgba(255,255,255,0.07)', 
    color='#94A3B8',
    tickfont=dict(color='#94A3B8')
)

# Forzamos los números externos a color claro
fig_vehiculos.update_traces(
    textposition='outside', 
    textfont=dict(size=11, color='#F1F5F9')
)

fig_vehiculos.show()

## 10. Conclusiones

**Resultados obtenidos y Fase de Experimentación:**
* Se construyeron dos modelos predictivos completos utilizando pipelines de scikit-learn, comparados de forma consistente sobre el mismo conjunto de test mediante métricas de error (RMSE, MAE y R²).
* Los datos climáticos históricos (extraídos de la API de Open-Meteo) y la matriz de feriados nacionales fueron incorporados con éxito como variables enriquecedoras del entorno.
* Se implementó un flujo de **automatización de experimentos con Papermill** para evaluar de manera sistemática la sensibilidad del Random Forest ante cambios en sus hiperparámetros (cantidad de estimadores y profundidad máxima) y su estabilidad frente a diferentes particiones de datos (semillas).
* Los resultados, predicciones y configuraciones de todos los escenarios evaluados fueron almacenados dinámicamente en colecciones de MongoDB Atlas.

**Interpretación de las hipótesis y Comportamiento del Modelo:**
* **H1 (Factor Climático):** La ganancia de información en el Random Forest validó empíricamente el rol crucial del entorno en la probabilidad de fatalidad. La `temp_media` y la `precipitacion` concentraron de forma combinada más del 62% de la importancia predictiva.
* **H2 (Efecto Calendario):** El análisis de probabilidad por día de la semana y la variable `es_feriado` contrastaron y validaron la estacionalidad del riesgo, confirmando un aumento marcado de la letalidad durante los días de descanso frente a la semana laboral estándar.
* **H3 (Rendimiento Algorítmico y Regularización):** La comparación de métricas refutó la premisa inicial. La Regresión Lineal obtuvo consistentemente una mayor capacidad de generalización. La fase de experimentación demostró que el Random Forest presentaba un sobreajuste severo frente a este conjunto de datos; y aunque la restricción de su profundidad y el aumento masivo de estimadores (de 50 a 200) lograron mitigar parte de la varianza, el ensamble nunca logró superar la estabilidad algorítmica del modelo paramétrico lineal.

**Posibles mejoras:**
* **Incorporar variables del factor humano y estructural:** Dado que los coeficientes de determinación ($R^2$) resultaron bajos en todos los escenarios evaluados, se evidencia que la siniestralidad vial es un fenómeno caótico y multifactorial. Incluir datos conductuales (exceso de velocidad, nivel de alcoholemia, uso de elementos de seguridad) y de infraestructura (estado de la calzada, iluminación) resulta indispensable para capturar la varianza que las variables macro de clima y calendario no logran explicar de forma aislada.
* Redefinir la variable objetivo como binaria (Siniestro Fatal / No Fatal) para evaluar el rendimiento mediante algoritmos de clasificación pura, aplicando técnicas de balanceo (como SMOTE).
* Optimizar la búsqueda de hiperparámetros mediante técnicas de validación cruzada estructurada, como `GridSearchCV`.

### Dashboard

In [140]:
import dash
from dash import dcc, html
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 0. PALETA Y ESTILOS GLOBALES
# ==========================================
COLORS = {
    'bg_dark':      '#0F1117',
    'bg_card':      '#1A1D27',
    'accent_blue':  '#4A90D9',
    'accent_teal':  '#2DD4BF',
    'accent_amber': '#F59E0B',
    'accent_coral': '#F87171',
    'text_primary': '#F1F5F9',
    'text_muted':   '#94A3B8',
    'border':       'rgba(255,255,255,0.07)',
}

CARD_STYLE = {
    'backgroundColor': COLORS['bg_card'],
    'borderRadius': '12px',
    'border': f"1px solid {COLORS['border']}",
    'padding': '24px',
}

TAB_STYLE = {
    'backgroundColor': COLORS['bg_dark'],
    'color': COLORS['text_primary'],
    'borderBottom': f"1px solid {COLORS['border']}",
    'border': 'none',
    'fontFamily': '"Inter", "Segoe UI", Arial, sans-serif',
    'fontWeight': '500',
    'fontSize': '14px',
    'padding': '12px 24px',
}

TAB_SELECTED_STYLE = {
    **TAB_STYLE,
    'backgroundColor': COLORS['bg_card'],
    'color': COLORS['accent_blue'],
    'borderTop': f"2px solid {COLORS['accent_blue']}",
    'borderBottom': 'none',
}

# ==========================================
# 1. KPIs
# ==========================================
fig_kpi = make_subplots(rows=1, cols=3, specs=[[{"type": "indicator"}, {"type": "indicator"}, {"type": "indicator"}]])

fig_kpi.add_trace(go.Indicator(
    mode="number", value=54064,
    title={"text": "Siniestros Analizados", "font": {"color": COLORS['text_primary'], "size": 13}},
    number={"font": {"color": COLORS['text_primary'], "size": 38}, "valueformat": ","},
), row=1, col=1)

fig_kpi.add_trace(go.Indicator(
    mode="number", value=0.98,
    number={"suffix": "%", "font": {"color": COLORS['accent_teal'], "size": 38}},
    title={"text": "Tasa de Fatalidad Media", "font": {"color": COLORS['text_primary'], "size": 13}},
), row=1, col=2)

fig_kpi.add_trace(go.Indicator(
    mode="number", value=0.0158,
    number={"valueformat": ".4f", "font": {"color": COLORS['accent_amber'], "size": 38}},
    title={"text": "Mejor R² — Reg. Lineal", "font": {"color": COLORS['text_primary'], "size": 13}},
), row=1, col=3)

fig_kpi.update_layout(paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', height=160, margin=dict(t=20, b=10, l=10, r=10))

# ==========================================
# 2. SISTEMA SEGURO Y TEMA OSCURO
# ==========================================
def placeholder(titulo):
    fig = go.Figure()
    fig.update_layout(
        paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', height=360,
        annotations=[dict(text=f"<b>{titulo}</b><br><span style='font-size:12px'>Falta ejecutar el gráfico</span>", 
                          x=0.5, y=0.5, showarrow=False, font=dict(size=15, color=COLORS['text_primary']))]
    )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return fig

def obtener_figura_segura(nombre_variable, titulo_placeholder):
    if nombre_variable in globals():
        return globals()[nombre_variable]
    return placeholder(titulo_placeholder)

# Incorporamos fig_real_predicho al ecosistema seguro
g_linea           = obtener_figura_segura('fig_linea',         'Evolución Histórica Temporal')
g_importancias    = obtener_figura_segura('fig_importancias',  'Top Variables Climáticas')
g_macro           = obtener_figura_segura('fig_macro',         'Impacto Feriados y Fines de Semana')
g_metricas        = obtener_figura_segura('fig_errores',       'Métricas de Error Algorítmico (RMSE/MAE)')
g_vehiculos       = obtener_figura_segura('fig_vehiculos',     'Vulnerabilidad por Tipo de Vehículo')
g_real_predicho   = obtener_figura_segura('fig_real_predicho', 'Real vs. Predicho (Conjunto Test)')

def dark_theme(fig, height=360):
    if not fig.layout.annotations or "Falta ejecutar" not in str(fig.layout.annotations):
        fig.update_layout(
            paper_bgcolor='rgba(0,0,0,0)', plot_bgcolor='rgba(0,0,0,0)', height=height, font_color=COLORS['text_primary'],
            margin=dict(t=40, b=10, l=10, r=10), 
            legend=dict(bgcolor='rgba(0,0,0,0)', font=dict(color=COLORS['text_primary']), orientation="h", yanchor="top", y=-0.2, xanchor="center", x=0.5),
            xaxis=dict(gridcolor=COLORS['border'], color=COLORS['text_primary'], automargin=True),
            yaxis=dict(gridcolor=COLORS['border'], color=COLORS['text_primary'], automargin=True),
        )
        
    return fig

# Aplicamos el tema oscuro a todos los gráficos, incluyendo el nuevo
for fig in [g_linea, g_importancias, g_macro, g_metricas, g_vehiculos, g_real_predicho]:
    dark_theme(fig)

# ==========================================
# 3. COMPONENTES RESPONSIVOS Y DE TEXTO
# ==========================================
FONT = '"Inter", "Segoe UI", Arial, sans-serif'

def section_title(text, subtitle=None):
    children = [html.H3(text, style={'color': COLORS['text_primary'], 'fontFamily': FONT, 'fontWeight': '600', 'fontSize': '18px', 'margin': '0 0 4px 0'})]
    if subtitle:
        children.append(html.P(subtitle, style={'color': COLORS['text_muted'], 'fontFamily': FONT, 'fontSize': '13px', 'margin': '0 0 20px 0'}))
    return html.Div(children, style={'marginBottom': '20px' if not subtitle else '0'})

def graph_card(fig, **extra_style):
    style = {**CARD_STYLE, **extra_style}
    return html.Div([dcc.Graph(figure=fig, config={'displayModeBar': False}, responsive=True)], style=style)

def two_col(left, right):
    return html.Div([left, right], style={'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(550px, 1fr))', 'gap': '24px', 'width': '100%'})

def conclusion_card(hipotesis, texto, valida=True):
    color_badge = COLORS['accent_teal'] if valida else COLORS['accent_coral']
    texto_badge = "HIPÓTESIS VALIDADA" if valida else "HIPÓTESIS REFUTADA"
    
    return html.Div([
        html.Div([
            html.Span(texto_badge, style={'backgroundColor': color_badge, 'color': '#0F1117', 'fontWeight': 'bold', 'padding': '4px 8px', 'borderRadius': '4px', 'fontSize': '11px', 'marginRight': '10px'}),
            html.Span(hipotesis, style={'color': COLORS['text_primary'], 'fontWeight': '600', 'fontSize': '15px'})
        ], style={'marginBottom': '12px'}),
        html.P(texto, style={'color': COLORS['text_muted'], 'fontSize': '14px', 'lineHeight': '1.6', 'margin': '0'})
    ], style={**CARD_STYLE, 'marginTop': '24px', 'borderLeft': f"4px solid {color_badge}"})


kpi_cards = html.Div([
    html.Div([html.P("Siniestros Analizados", style={'color': COLORS['text_muted'], 'fontFamily': FONT, 'fontSize': '12px', 'margin': '0 0 6px 0'}),
              html.P("54.064", style={'color': COLORS['text_primary'], 'fontFamily': FONT, 'fontSize': '28px', 'fontWeight': '700', 'margin': '0'})], style={**CARD_STYLE, 'borderLeft': f"3px solid {COLORS['accent_blue']}"}),
    html.Div([html.P("Tasa de Fatalidad Media", style={'color': COLORS['text_muted'], 'fontFamily': FONT, 'fontSize': '12px', 'margin': '0 0 6px 0'}),
              html.P("0,98 %", style={'color': COLORS['accent_teal'], 'fontFamily': FONT, 'fontSize': '28px', 'fontWeight': '700', 'margin': '0'})], style={**CARD_STYLE, 'borderLeft': f"3px solid {COLORS['accent_teal']}"}),
    html.Div([html.P("Mejor R² — Regresión Lineal", style={'color': COLORS['text_muted'], 'fontFamily': FONT, 'fontSize': '12px', 'margin': '0 0 6px 0'}),
              html.P("0.0158", style={'color': COLORS['accent_amber'], 'fontFamily': FONT, 'fontSize': '28px', 'fontWeight': '700', 'margin': '0'})], style={**CARD_STYLE, 'borderLeft': f"3px solid {COLORS['accent_amber']}"}),
], style={'display': 'grid', 'gridTemplateColumns': 'repeat(auto-fit, minmax(280px, 1fr))', 'gap': '16px', 'marginBottom': '24px'})

# ==========================================
# 4. LAYOUT FINAL
# ==========================================
app = dash.Dash(__name__)

app.layout = html.Div(style={'backgroundColor': COLORS['bg_dark'], 'minHeight': '100vh', 'padding': '32px 40px', 'fontFamily': FONT, 'boxSizing': 'border-box'}, children=[
    html.Div([
        html.Div([
            html.Span("The Outliers", style={'backgroundColor': COLORS['accent_blue'], 'color': '#fff', 'fontFamily': FONT, 'fontSize': '11px', 'fontWeight': '600', 'padding': '4px 10px', 'borderRadius': '4px', 'marginBottom': '10px', 'display': 'inline-block'}),
            html.H1("Siniestralidad Vial", style={'color': COLORS['text_primary'], 'fontFamily': FONT, 'fontWeight': '700', 'fontSize': '32px', 'margin': '8px 0 4px 0'}),
            html.P("Análisis exploratorio y modelado predictivo — Argentina", style={'color': COLORS['text_muted'], 'fontFamily': FONT, 'fontSize': '15px', 'margin': '0'}),
        ]),
    ], style={'marginBottom': '32px', 'borderBottom': f"1px solid {COLORS['border']}", 'paddingBottom': '24px'}),
    
    kpi_cards,
    
    dcc.Tabs(style={'backgroundColor': COLORS['bg_dark']}, children=[
        
        # --- PESTAÑA 1 --- (Actualizada con los dos gráficos descriptivos)
        dcc.Tab(label='Resumen general', style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE, children=[
            html.Div([
                section_title("Evolución histórica de siniestros", "Serie temporal completa del período analizado (2019-2024)"),
                graph_card(g_linea),
                
                html.Div(style={'height': '24px'}), # Espaciador
                
                section_title("Perfil de Vulnerabilidad", "Tasa de letalidad media según el modo de desplazamiento de la víctima"),
                graph_card(g_vehiculos),
            ], style={'marginTop': '24px'}),
        ]),

        # --- PESTAÑA 2 ---
        dcc.Tab(label='Clima y Calendario', style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE, children=[
            html.Div([
                html.Div([
                    html.Span("H1", style={'backgroundColor': COLORS['accent_teal'], 'color': '#0F1117', 'fontSize': '11px', 'fontWeight': '700', 'padding': '2px 8px', 'borderRadius': '4px', 'marginRight': '10px'}),
                    html.Span("Variables Climáticas  ·  ", style={'color': COLORS['text_primary'], 'fontWeight': '600'}),
                    html.Span("H2", style={'backgroundColor': COLORS['accent_amber'], 'color': '#0F1117', 'fontSize': '11px', 'fontWeight': '700', 'padding': '2px 8px', 'borderRadius': '4px', 'marginRight': '10px'}),
                    html.Span("Feriados y Fines de Semana", style={'color': COLORS['text_primary'], 'fontWeight': '600'}),
                ], style={'marginTop': '24px', 'marginBottom': '20px'}),
                
                two_col(graph_card(g_importancias), graph_card(g_macro)),
                
                # Cajas de Conclusión para H1 y H2
                two_col(
                    conclusion_card(
                        "Hipótesis 1: El factor climático incide en la letalidad.", 
                        "Las variables meteorológicas (temperatura media y precipitaciones) concentran más del 62% de la importancia predictiva en el modelo de Random Forest, demostrando empíricamente que el contexto ambiental es un determinante clave en la gravedad de un siniestro.", 
                        valida=True
                    ),
                    conclusion_card(
                        "Hipótesis 2: Efecto Calendario (Fin de Semana/Feriados).", 
                        "Los datos confirman una marcada estacionalidad. La probabilidad de fatalidad se incrementa drásticamente durante los días de descanso, alcanzando su pico histórico de letalidad media los días domingo frente a los días laborables.", 
                        valida=True
                    )
                )
            ])
        ]),

        # --- PESTAÑA 3 --- (Actualizada con Real vs Predicho)
        dcc.Tab(label='Evaluación Algorítmica', style=TAB_STYLE, selected_style=TAB_SELECTED_STYLE, children=[
            html.Div([
                html.Div([
                    html.Span("H3", style={'backgroundColor': COLORS['accent_coral'], 'color': '#0F1117', 'fontSize': '11px', 'fontWeight': '700', 'padding': '2px 8px', 'borderRadius': '4px', 'marginRight': '10px'}),
                    html.Span("¿Random Forest superó a la Regresión Lineal?", style={'color': COLORS['text_primary'], 'fontWeight': '600'}),
                ], style={'marginTop': '24px', 'marginBottom': '20px'}),
                
                # Ahora mostramos Métricas de Error junto al gráfico Real vs Predicho
                two_col(graph_card(g_metricas), graph_card(g_real_predicho)),
                
                # Caja de Conclusión para H3
                conclusion_card(
                    "Hipótesis 3: El modelo de Ensamble (RF) superará al Paramétrico (LR).", 
                    "Contrario a lo esperado matemáticamente, la Regresión Lineal Múltiple demostró una mayor capacidad de generalización en el conjunto de prueba (R² = 0.0158). El algoritmo de Random Forest, a pesar de capturar relaciones no lineales, sufrió de sobreajuste (overfitting) arrojando un coeficiente de determinación negativo.", 
                    valida=False
                )
            ])
        ]),
    ])
])

if __name__ == '__main__':
    app.run(jupyter_mode="external", port=8050)

Dash app running on http://127.0.0.1:8050/
